# 🔍 LIR Motif Finder
### Find LC3-Interacting Region (LIR) motifs in Intrinsically Disordered Regions

---

This notebook:
1. Predicts disordered regions in your protein sequences using **metapredict V2**
2. Scans those regions for **LIR motifs** (the docking sites for ATG8/LC3/GABARAP proteins in autophagy)
3. Outputs a results table and saves a CSV file

**Motif classes searched:**

| Type | Pattern | Description |
|------|---------|-------------|
| `basic` | `[WFY]xx[LIV]` | Canonical LIR core |
| `extended` | `[DE][WFY]xx[LIV]` | Single upstream acidic residue |
| `acidic_extended` | `[DE]{2}[WFY]xx[LIV]` | Two upstream acidic residues |

> **Tip:** Run cells top to bottom. Only **Cell 1** (setup) and **Cell 2** (your inputs) need editing.

---
## Cell 0 — Install dependencies
Run this once. You can skip it if you already have the packages installed.

In [ ]:
 !pip install metapredict biopython pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 3.1 MB/s  0:00:04m0:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached cython-3.2.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (7.5 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1

---
## Cell 1 — Imports & core functions
Run this cell first — it loads all the logic.

In [ ]:
import re
import sys
import warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Iterator

warnings.filterwarnings("ignore")

# ── BioPython (optional, but recommended) ──────────────────────────────────
try:
    from Bio import SeqIO
    _BIOPYTHON = True
except ImportError:
    _BIOPYTHON = False

# ── metapredict ─────────────────────────────────────────────────────────────
try:
    import metapredict as meta
    print("✅ metapredict loaded")
except ImportError:
    print("❌ metapredict not found. Run: pip install metapredict")
    sys.exit()

# ── pandas ───────────────────────────────────────────────────────────────────
try:
    import pandas as pd
    print("✅ pandas loaded")
except ImportError:
    print("❌ pandas not found. Run: pip install pandas")
    sys.exit()

print("✅ All imports successful — ready to run!")


# ═══════════════════════════════════════════════════════════════════════════
#  Motif patterns
# ═══════════════════════════════════════════════════════════════════════════
_PATTERNS = {
    "basic":           re.compile(r"(?=([WFY][A-Z]{2}[LIV]))"),
    "extended":        re.compile(r"(?=([DE][WFY][A-Z]{2}[LIV]))"),
    "acidic_extended": re.compile(r"(?=([DE]{2}[WFY][A-Z]{2}[LIV]))"),
}


# ═══════════════════════════════════════════════════════════════════════════
#  Built-in FASTA parser (fallback if BioPython is absent)
# ═══════════════════════════════════════════════════════════════════════════
class _Record:
    __slots__ = ("id", "description", "seq")
    def __init__(self, header, seq):
        parts = header.split(None, 1)
        self.id = parts[0]
        self.description = header
        self.seq = seq

def _parse_fasta(path):
    header, chunks = "", []
    with open(path) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith(">"):
                if header:
                    yield _Record(header, "".join(chunks))
                header, chunks = line[1:], []
            else:
                chunks.append(line)
    if header:
        yield _Record(header, "".join(chunks))

def _iter_fasta(path):
    if _BIOPYTHON:
        return SeqIO.parse(path, "fasta")
    return _parse_fasta(path)

def _seq(record):
    """Get sequence string regardless of record type."""
    return str(record.seq).upper()


# ═══════════════════════════════════════════════════════════════════════════
#  Disorder prediction
# ═══════════════════════════════════════════════════════════════════════════
def get_disordered_regions(sequence, threshold, min_length):
    """
    Run metapredict and return list of (start, end, subsequence) tuples.
    Positions are 0-based, end is exclusive.
    """
    scores = meta.predict_disorder(sequence)
    mask = [s >= threshold for s in scores]
    regions, i, n = [], 0, len(sequence)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:
                j += 1
            if (j - i) >= min_length:
                regions.append((i, j, sequence[i:j]))
            i = j
        else:
            i += 1
    return scores, regions


# ═══════════════════════════════════════════════════════════════════════════
#  Main analysis function
# ═══════════════════════════════════════════════════════════════════════════
def run_analysis(fasta_path, threshold=0.5, min_disorder=5, motif_types=None):
    """
    Analyse all proteins in a FASTA file.
    Returns a list of result dicts (one row per motif hit).
    """
    fasta_path = str(fasta_path)
    active_types = motif_types or list(_PATTERNS.keys())
    rows = []

    for record in _iter_fasta(fasta_path):
        seq = _seq(record)
        pid = record.id
        desc = record.description

        scores, regions = get_disordered_regions(seq, threshold, min_disorder)

        for (reg_start, reg_end, reg_seq) in regions:
            for mtype in active_types:
                for match in _PATTERNS[mtype].finditer(reg_seq):
                    motif_seq = match.group(1)
                    local_start = match.start()
                    prot_start  = reg_start + local_start          # 0-based
                    prot_end    = prot_start + len(motif_seq)       # exclusive

                    motif_scores = scores[prot_start:prot_end]
                    mean_score = round(
                        sum(motif_scores) / len(motif_scores), 4
                    ) if motif_scores else 0.0

                    rows.append({
                        "protein_id":              pid,
                        "protein_description":     desc,
                        "protein_length":          len(seq),
                        "motif_type":              mtype,
                        "motif_sequence":          motif_seq,
                        "motif_start (1-based)":   prot_start + 1,
                        "motif_end (1-based)":     prot_end,
                        "IDR_start (1-based)":     reg_start + 1,
                        "IDR_end (1-based)":       reg_end,
                        "IDR_length":              reg_end - reg_start,
                        "mean_disorder_score":     mean_score,
                    })

    return rows

---
## Cell 2 — ✏️ Set your inputs here

**This is the only cell you need to edit.**  
Set `FASTA_FILE` to the path of your FASTA file, then adjust the parameters if needed.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  👇  EDIT THIS
# ─────────────────────────────────────────────────────────────────────────────

FASTA_FILE = "proteins.fasta"   # ← put your FASTA file path here
                                 #   (absolute or relative to this notebook)

# ─────────────────────────────────────────────────────────────────────────────
#  Optional parameters (safe to leave as-is)
# ─────────────────────────────────────────────────────────────────────────────

DISORDER_THRESHOLD  = 0.5   # metapredict score cutoff (0–1). Higher = stricter disorder.
                             # Recommended range: 0.45 – 0.6

MIN_DISORDER_LENGTH = 5     # Minimum IDR length (residues) to consider.
                             # Shorter IDRs are ignored.

MOTIF_TYPES = [             # Which motif classes to search. Remove any you don't need.
    "basic",                #   [WFY]xx[LIV]
    "extended",             #   [DE][WFY]xx[LIV]
    "acidic_extended",      #   [DE]{2}[WFY]xx[LIV]
]

OUTPUT_CSV = "lir_hits.csv"  # Results will be saved here

# ─────────────────────────────────────────────────────────────────────────────

# Validate file path
if not Path(FASTA_FILE).exists():
    print(f"❌  File not found: {FASTA_FILE}")
    print("    Please check the path and try again.")
else:
    print(f"✅  FASTA file found: {FASTA_FILE}")
    print(f"    Disorder threshold : {DISORDER_THRESHOLD}")
    print(f"    Min IDR length     : {MIN_DISORDER_LENGTH} residues")
    print(f"    Motif types        : {', '.join(MOTIF_TYPES)}")
    print(f"    Output CSV         : {OUTPUT_CSV}")

---
## Cell 3 — Run the analysis

In [ ]:
print("🔍 Running analysis...\n")

results = run_analysis(
    fasta_path       = FASTA_FILE,
    threshold        = DISORDER_THRESHOLD,
    min_disorder     = MIN_DISORDER_LENGTH,
    motif_types      = MOTIF_TYPES,
)

if results:
    df = pd.DataFrame(results)
    print(f"✅  Done!")
    print(f"    Proteins with hits : {df['protein_id'].nunique()}")
    print(f"    Total LIR hits     : {len(df)}")
    print()
    print("Breakdown by motif type:")
    print(df["motif_type"].value_counts().to_string())
else:
    df = pd.DataFrame()
    print("⚠️  No LIR motifs found in disordered regions with the current settings.")
    print("    Try lowering DISORDER_THRESHOLD or MIN_DISORDER_LENGTH.")

---
## Cell 4 — View results table

In [ ]:
if df.empty:
    print("No results to display.")
else:
    # Pretty display — truncate long descriptions
    display_df = df.copy()
    display_df["protein_description"] = display_df["protein_description"].str[:60] + "..."
    
    pd.set_option("display.max_rows", 50)
    pd.set_option("display.max_colwidth", 70)
    pd.set_option("display.width", 120)
    
    display(display_df)

---
## Cell 5 — Filter results (optional)
Use these quick filters to narrow down hits of interest.

In [ ]:
if df.empty:
    print("No results to filter.")
else:
    # ── Adjust any of these filters (set to None to disable) ──────────────
    FILTER_MOTIF_TYPE    = None          # e.g. "basic" or "acidic_extended"
    FILTER_PROTEIN_ID    = None          # e.g. "sp|P12345|ATG13_HUMAN"
    MIN_DISORDER_SCORE   = 0.6           # Only show motifs with mean disorder ≥ this value
    MIN_IDR_LENGTH       = None          # e.g. 20 — only show motifs in longer IDRs
    # ───────────────────────────────────────────────────────────────────────

    filtered = df.copy()

    if FILTER_MOTIF_TYPE:
        filtered = filtered[filtered["motif_type"] == FILTER_MOTIF_TYPE]
    if FILTER_PROTEIN_ID:
        filtered = filtered[filtered["protein_id"].str.contains(FILTER_PROTEIN_ID, case=False)]
    if MIN_DISORDER_SCORE is not None:
        filtered = filtered[filtered["mean_disorder_score"] >= MIN_DISORDER_SCORE]
    if MIN_IDR_LENGTH is not None:
        filtered = filtered[filtered["IDR_length"] >= MIN_IDR_LENGTH]

    print(f"Filtered hits: {len(filtered)} (from {len(df)} total)")
    display(filtered.reset_index(drop=True))

---
## Cell 6 — Save results to CSV

In [ ]:
if df.empty:
    print("No results to save.")
else:
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅  Results saved to: {OUTPUT_CSV}")
    print(f"    {len(df)} rows written.")

---
## Cell 7 — Summary statistics

In [ ]:
if df.empty:
    print("No results to summarise.")
else:
    print("═" * 55)
    print("  LIR Motif Finder — Summary")
    print("═" * 55)
    print(f"  FASTA file             : {FASTA_FILE}")
    print(f"  Disorder threshold     : {DISORDER_THRESHOLD}")
    print(f"  Min IDR length         : {MIN_DISORDER_LENGTH} residues")
    print()
    print(f"  Proteins with ≥1 hit   : {df['protein_id'].nunique()}")
    print(f"  Total LIR hits         : {len(df)}")
    print()
    print("  Hits by motif type:")
    for mtype, count in df["motif_type"].value_counts().items():
        print(f"    {mtype:<20}: {count}")
    print()
    print("  Unique motif sequences (top 10):")
    for seq, count in df["motif_sequence"].value_counts().head(10).items():
        print(f"    {seq}  ×{count}")
    print()
    print("  Disorder score stats (at motif positions):")
    print(df["mean_disorder_score"].describe().rename({
        "count": "n", "mean": "mean", "std": "std",
        "min": "min", "25%": "Q1", "50%": "median", "75%": "Q3", "max": "max"
    }).to_string())
    print()
    print("  Top 10 proteins by hit count:")
    top = df.groupby("protein_id").size().sort_values(ascending=False).head(10)
    for pid, cnt in top.items():
        print(f"    {pid:<30}  {cnt} hit(s)")
    print("═" * 55)

---
## Cell 8 — Visualisation (optional)
Quick plots to explore the results.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams.update({"figure.dpi": 110, "font.size": 11})
    _MATPLOTLIB = True
except ImportError:
    print("matplotlib not installed — skipping plots. Run: pip install matplotlib")
    _MATPLOTLIB = False

if _MATPLOTLIB and not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # ── Plot 1: Hits by motif type ─────────────────────────────────────────
    type_counts = df["motif_type"].value_counts()
    axes[0].bar(type_counts.index, type_counts.values,
                color=["#4C72B0", "#DD8452", "#55A868"][:len(type_counts)])
    axes[0].set_title("Hits by motif type")
    axes[0].set_ylabel("Count")
    axes[0].set_xlabel("Motif type")

    # ── Plot 2: Distribution of mean disorder scores at motif positions ─────
    axes[1].hist(df["mean_disorder_score"], bins=20,
                 color="#4C72B0", edgecolor="white")
    axes[1].axvline(DISORDER_THRESHOLD, color="red", linestyle="--",
                    label=f"threshold ({DISORDER_THRESHOLD})")
    axes[1].set_title("Disorder scores at motif positions")
    axes[1].set_xlabel("Mean disorder score")
    axes[1].set_ylabel("Count")
    axes[1].legend()

    # ── Plot 3: Distribution of containing IDR lengths ─────────────────────
    axes[2].hist(df["IDR_length"], bins=20,
                 color="#55A868", edgecolor="white")
    axes[2].set_title("IDR lengths containing LIR motifs")
    axes[2].set_xlabel("IDR length (residues)")
    axes[2].set_ylabel("Count")

    plt.tight_layout()
    plt.savefig("lir_motif_summary_plots.png", bbox_inches="tight")
    plt.show()
    print("📊 Plots saved to: lir_motif_summary_plots.png")